## 0 · Cài đặt thư viện

In [1]:

!pip install pyarrow pandas tqdm


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:

import os
import sys
import gc
import json
import shutil
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import psutil
from tqdm.auto import tqdm 
import matplotlib.pyplot as plt
import seaborn as sns


pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 150)
pd.set_option('display.float_format', lambda x: '%.4f' % x) 


ROOT_DIR = Path().resolve().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))
TARGET_CATEGORY = "Home_and_Kitchen"
def ram_usage():
    vm = psutil.virtual_memory()
    return f"RAM: {vm.used/(1024**3):.1f}/{vm.total/(1024**3):.1f} GB ({vm.percent:.0f}%)"
print(f"Trạng thái hệ thống: {ram_usage()}")

Trạng thái hệ thống: RAM: 5.6/7.8 GB (71%)


## 1 · Config & paths

In [3]:
from pathlib import Path


REVIEW_PATH = Path(r"D:\Documents\Project\Data Mining\Home_and_Kitchen.jsonl\Home_and_Kitchen.jsonl")
META_PATH   = Path(r"D:\Documents\Project\Data Mining\meta_Home_and_Kitchen.jsonl\meta_Home_and_Kitchen.jsonl")

OUT_DIR        = Path(r"D:\\Documents\\Project\\Data Mining\data\processed")
REVIEW_OUT_DIR = OUT_DIR / "reviews"
META_OUT_DIR   = OUT_DIR / "meta"

REVIEW_OUT_DIR.mkdir(parents=True, exist_ok=True)
META_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Review path exists :", REVIEW_PATH.exists())
print("Meta   path exists :", META_PATH.exists())
print(f"Review output      : {REVIEW_OUT_DIR}")
print(f"Meta   output      : {META_OUT_DIR}")
CHUNK_SIZE      = 200_000          
TARGET_CATEGORY = "Home_and_Kitchen"  
for name, path in [("Review", REVIEW_PATH), ("Meta", META_PATH)]:
    size_gb = path.stat().st_size / (1024**3) if path.exists() else 0
    status  = f"Tìm được ({size_gb:.2f} GB)" if path.exists() else "không thấy"
    print(f"  {name}: {status}")
    print(f"        Path: {path}")


Review path exists : True
Meta   path exists : True
Review output      : D:\Documents\Project\Data Mining\data\processed\reviews
Meta   output      : D:\Documents\Project\Data Mining\data\processed\meta
  Review: Tìm được (29.25 GB)
        Path: D:\Documents\Project\Data Mining\Home_and_Kitchen.jsonl\Home_and_Kitchen.jsonl
  Meta: Tìm được (10.98 GB)
        Path: D:\Documents\Project\Data Mining\meta_Home_and_Kitchen.jsonl\meta_Home_and_Kitchen.jsonl


## 2. Explore Data



In [4]:
import json
print('\n review_Home_and_Kitchen.jsonl')
with open(REVIEW_PATH, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        data = json.loads(line)
        print(f"\n--- Dòng {i+1} ---")
        for key, value in data.items():
            display_val = str(value)[:80] + "..." if len(str(value)) > 80 else value
            print(f"  [{key}]: {display_val}")




 review_Home_and_Kitchen.jsonl

--- Dòng 1 ---
  [rating]: 1.0
  [title]: Received Used & scratched item! Purchased new!
  [text]: Livid.  Once again received an obviously used item that has food on it & scratch...
  [images]: []
  [asin]: B007WQ9YNO
  [parent_asin]: B09XWYG6X1
  [user_id]: AFKZENTNBQ7A7V7UXW5JJI6UGRYQ
  [timestamp]: 1677373409298
  [helpful_vote]: 1
  [verified_purchase]: True

--- Dòng 2 ---
  [rating]: 5.0
  [title]: Excellent for moving & storage & floods!
  [text]: I purchased these for multiple reasons. The main reason was that I was moving. I...
  [images]: []
  [asin]: B09H2VJW6K
  [parent_asin]: B0BXDLF8TW
  [user_id]: AFKZENTNBQ7A7V7UXW5JJI6UGRYQ
  [timestamp]: 1672043410846
  [helpful_vote]: 0
  [verified_purchase]: True

--- Dòng 3 ---
  [rating]: 2.0
  [title]: Lid very loose- needs a gasket imo. Small base.
  [text]: [[VIDEOID:c87e962bc893a948856b0f1b285ce6cc]] I wanted to love this bc I previous...
  [images]: [{'small_image_url': 'https://m.media-amazo

In [5]:
print('\n meta_Home_and_Kitchen.jsonl')
with open(META_PATH, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        data = json.loads(line)
        print(f"\n--- Dòng {i+1} ---")
        for key, value in data.items():
            display_val = str(value)[:80] + "..." if len(str(value)) > 80 else value
            print(f"  [{key}]: {display_val}")


 meta_Home_and_Kitchen.jsonl

--- Dòng 1 ---
  [main_category]: Amazon Home
  [title]: Set of 4 Irish Coffee Glass Mugs Footed 10.5 oz.Thick Wall Glass For Coffee, tea...
  [average_rating]: 4.6
  [rating_number]: 18
  [features]: ['☕PERFECT IRISH COFFEE MUG: With our clear glass 10.5 ounce makes it a perfect ...
  [description]: ['Set of 12 Footed 10.5 oz. Irish coffee mug the perfect vessel for offering up ...
  [price]: 24.95
  [images]: [{'thumb': 'https://m.media-amazon.com/images/I/41zOU0mN04L._AC_US75_.jpg', 'lar...
  [videos]: [{'title': 'Irish Coffee Glass Coffee Mugs Regal Shape 8 oz. Cappuccinos', 'url'...
  [store]: LavoHome
  [categories]: ['Home & Kitchen', 'Kitchen & Dining', 'Dining & Entertaining', 'Glassware & Dri...
  [details]: {'Brand': 'LavoHome', 'Material': 'Glass', 'Color': 'Clear', 'Capacity': '10.5 O...
  [parent_asin]: B07R3DYMH6
  [bought_together]: None

--- Dòng 2 ---
  [main_category]: Amazon Home
  [title]: Foaming Soap Dispenser Thick Ceramic Foam Han

In [6]:
def peek_jsonl_gz(filepath, n_rows=1000):
    data = []
    with open(filepath, 'rt', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= n_rows:
                break
            try:
                data.append(json.loads(line.strip()))
            except json.JSONDecodeError:
                continue  
    return pd.DataFrame(data)

df_review_peek = peek_jsonl_gz(REVIEW_PATH, n_rows=1000)
df_meta_peek   = peek_jsonl_gz(META_PATH,   n_rows=1000)

print(f"Review sample shape : {df_review_peek.shape}")
print(f"Meta sample shape   : {df_meta_peek.shape}")

Review sample shape : (1000, 10)
Meta sample shape   : (1000, 14)


In [7]:
print("\nColumn list of Review File:")
print(df_review_peek.columns.tolist())

print("\n Data types and missing values:")
missing_info = pd.DataFrame({
    'dtype'        : df_review_peek.dtypes,
    'non_null'     : df_review_peek.count(),
    'missing'      : df_review_peek.isnull().sum(),
    'missing_%'    : (df_review_peek.isnull().sum() / len(df_review_peek) * 100).round(2)
})
print(missing_info)
display(df_review_peek.head(3))


Column list of Review File:
['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

 Data types and missing values:
                     dtype  non_null  missing  missing_%
rating             float64      1000        0     0.0000
title               object      1000        0     0.0000
text                object      1000        0     0.0000
images              object      1000        0     0.0000
asin                object      1000        0     0.0000
parent_asin         object      1000        0     0.0000
user_id             object      1000        0     0.0000
timestamp            int64      1000        0     0.0000
helpful_vote         int64      1000        0     0.0000
verified_purchase     bool      1000        0     0.0000


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,1.0000,Received Used & scratched item! Purchased new!,Livid. Once again received an obviously used item that has food on it & scratches. I purchased this new!! Pics not loading rn. Will add them lat...,[],B007WQ9YNO,B09XWYG6X1,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1677373409298,1,True
1,5.0000,Excellent for moving & storage & floods!,I purchased these for multiple reasons. The main reason was that I was moving. I was moving bc my apt kept flooding. Luckily having been through ...,[],B09H2VJW6K,B0BXDLF8TW,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1672043410846,0,True
2,2.0000,Lid very loose- needs a gasket imo. Small base.,"[[VIDEOID:c87e962bc893a948856b0f1b285ce6cc]] I wanted to love this bc I previously bought a matching turquoise teapot, but the loose lid (defectiv...","[{'small_image_url': 'https://m.media-amazon.com/images/I/616kZbRGDpL._SL256_.jpg', 'medium_image_url': 'https://m.media-amazon.com/images/I/616kZ...",B07RL297VR,B09G2PW8ZG,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1653447296788,0,True


In [8]:
print("\n Column list of meta file:")
print(df_meta_peek.columns.tolist())

print("\n Data types and missing values:")
missing_info_meta = pd.DataFrame({
    'dtype'     : df_meta_peek.dtypes,
    'non_null'  : df_meta_peek.count(),
    'missing'   : df_meta_peek.isnull().sum(),
    'missing_%' : (df_meta_peek.isnull().sum() / len(df_meta_peek) * 100).round(2)
})
print(missing_info_meta)
display(df_meta_peek.head(3))


 Column list of meta file:
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together']

 Data types and missing values:
                   dtype  non_null  missing  missing_%
main_category     object       964       36     3.6000
title             object      1000        0     0.0000
average_rating   float64      1000        0     0.0000
rating_number      int64      1000        0     0.0000
features          object      1000        0     0.0000
description       object      1000        0     0.0000
price            float64       537      463    46.3000
images            object      1000        0     0.0000
videos            object      1000        0     0.0000
store             object       987       13     1.3000
categories        object      1000        0     0.0000
details           object      1000        0     0.0000
parent_asin       object      1000    

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Amazon Home,"Set of 4 Irish Coffee Glass Mugs Footed 10.5 oz.Thick Wall Glass For Coffee, tea, Cappuccinos, Mulled Ciders,Hot Chocolates, Ice cream and More",4.6000,18,"[☕PERFECT IRISH COFFEE MUG: With our clear glass 10.5 ounce makes it a perfect mug for offering up your signature Coffee Drinks, Cappuccinos, Milk...","[Set of 12 Footed 10.5 oz. Irish coffee mug the perfect vessel for offering up your signature cappuccinos, mulled ciders, hot chocolates, and more...",24.9500,"[{'thumb': 'https://m.media-amazon.com/images/I/41zOU0mN04L._AC_US75_.jpg', 'large': 'https://m.media-amazon.com/images/I/41zOU0mN04L._AC_.jpg', '...","[{'title': 'Irish Coffee Glass Coffee Mugs Regal Shape 8 oz. Cappuccinos', 'url': 'https://www.amazon.com/vdp/00df5d3c292b485fa88aa6c4537dcad5?ref...",LavoHome,"[Home & Kitchen, Kitchen & Dining, Dining & Entertaining, Glassware & Drinkware, Cups, Mugs, & Saucers, Mug Sets]","{'Brand': 'LavoHome', 'Material': 'Glass', 'Color': 'Clear', 'Capacity': '10.5 Ounces', 'Style': 'Modern', 'Pattern': 'Solid', 'Product Care Instr...",B07R3DYMH6,None
1,Amazon Home,"Foaming Soap Dispenser Thick Ceramic Foam Hand Soap Dispenser for Bathroom or Kitchen Sink, Liquid Pump Bottles for Hand soap, Body Wash, 2 Pack B...",4.4000,135,[Saving money: You can DIY foam soap which will save you hundreds of dollars ( create foam just need to dilute the regular soap with water in a ra...,[],24.9900,"[{'thumb': 'https://m.media-amazon.com/images/I/31n27bc-h9L._AC_US75_.jpg', 'large': 'https://m.media-amazon.com/images/I/31n27bc-h9L._AC_.jpg', '...","[{'title': 'Foaming Soap Dispenser Ceramic Foaming Hand Soap Dispenser', 'url': 'https://www.amazon.com/vdp/057a50cebe7848d4bb025e648e934c62?ref=d...",rejomiik,"[Home & Kitchen, Bath]","{'Package Dimensions': '7.32 x 6.14 x 3.94 inches', 'Item Weight': '1.82 pounds', 'Best Sellers Rank': {'Home & Kitchen': 47994, 'Bath Products': ...",B0BNZ8Q7YT,None
2,Amazon Home,"Tapestry Trading 558W90 90 in. European Lace Table Cloth, White",5.0000,3,"[Polyester,lace, European Lace Tablecloth, 100% Polyester, Machine Washable, Shape - Round, Dryer Dryable]",[Features. European Lace Tablecloth. 100 Polyester. Machine Washable. Shape - Round. Dryer Dryable. Color - White. Size - 90 in.],45.6400,"[{'thumb': 'https://m.media-amazon.com/images/I/51Po9mm+cKL._AC_US75_.jpg', 'large': 'https://m.media-amazon.com/images/I/51Po9mm+cKL._AC_.jpg', '...",[],Tapestry Trading,"[Home & Kitchen, Kitchen & Dining, Kitchen & Table Linens, Tablecloths]","{'Brand': 'Tapestry Trading', 'Color': 'White', 'Material': 'Polyester', 'Shape': 'Round', 'Care instructions': 'Machine Wash', 'Fabric Type': 'Po...",B01508WQC6,None


## 3 · Data collection

In [9]:
import psutil
from tqdm import tqdm

def ram_usage():
    vm = psutil.virtual_memory()
    return f"RAM: {vm.used/(1024**3):.1f}/{vm.total/(1024**3):.1f} GB ({vm.percent:.0f}%)"
print(f"{ram_usage()}")

RAM: 5.6/7.8 GB (72%)


In [10]:
import numpy as np
from pathlib import Path


DATA_DIR      = Path(r"D:/Documents/Project/Data Mining") 
REVIEW_PATH   = Path(r"D:\Documents\Project\Data Mining\Home_and_Kitchen.jsonl\Home_and_Kitchen.jsonl") 
META_PATH     = Path(r"D:\Documents\Project\Data Mining\meta_Home_and_Kitchen.jsonl\meta_Home_and_Kitchen.jsonl") 
ROOT_DIR      = Path(r"D:/Documents/Project/Data Mining")
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
SAMPLE_DIR    = ROOT_DIR / "data" / "sample"
FIGURES_DIR   = ROOT_DIR / "outputs" / "figures"
TEMP_DIR      = ROOT_DIR / "data" / "processed" / "_temp_chunks"

for d in [PROCESSED_DIR, SAMPLE_DIR, FIGURES_DIR, TEMP_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    
CHUNK_SIZE           = 200_000
MIN_TEXT_LENGTH      = 10
MIN_REVIEWS_PER_USER = 5
MIN_REVIEWS_PER_ITEM = 5
RANDOM_SEED          = 42

np.random.seed(RANDOM_SEED)

print("=== CẤU HÌNH & THÔNG TIN FILE ===")
print(f"   Chunk size : {CHUNK_SIZE:,} dòng/lần")

if REVIEW_PATH.exists():
    print(f"   Review file: {REVIEW_PATH.stat().st_size/(1024**3):.2f} GB")
else:
    print(f"   Review file: Không tìm thấy tại {REVIEW_PATH}")

if META_PATH.exists():
    print(f"   Meta file  : {META_PATH.stat().st_size/(1024**3):.2f} GB")
else:
    print(f"   Meta file  : Không tìm thấy tại {META_PATH}")

print(f"   RAM: {ram_usage()}")

=== CẤU HÌNH & THÔNG TIN FILE ===
   Chunk size : 200,000 dòng/lần
   Review file: 29.25 GB
   Meta file  : 10.98 GB
   RAM: RAM: 5.6/7.8 GB (72%)


In [11]:
def count_lines(filepath):
    count = 0
    with open(filepath, 'rt', encoding='utf-8') as f:
        for _ in tqdm(f, desc=filepath.name[:35], unit=" lines"):
            count += 1
    return count

n_reviews = count_lines(REVIEW_PATH)
n_meta    = count_lines(META_PATH)
n_chunks  = (n_reviews // CHUNK_SIZE) + 1

print(f"   Review: {n_reviews:,} dòng → {n_chunks} chunks")
print(f"   Meta  : {n_meta:,} dòng")

Home_and_Kitchen.jsonl: 67409944 lines [00:48, 1389477.43 lines/s]
meta_Home_and_Kitchen.jsonl: 3735584 lines [00:13, 270194.52 lines/s]

   Review: 67,409,944 dòng → 338 chunks
   Meta  : 3,735,584 dòng


In [12]:

def clean_review_chunk(df):
    if df.empty: return None
    
    needed = ['rating', 'text', 'title', 'user_id', 'parent_asin', 'timestamp', 'verified_purchase']
    df = df[[c for c in needed if c in df.columns]].copy()
    df = df.dropna(subset=['rating', 'text'])
    if df.empty: return None
    df['rating'] = pd.to_numeric(df['rating'], errors='coerce').astype('float32')
    df = df.dropna(subset=['rating'])
    
    if 'verified_purchase' in df.columns:
        df['verified_purchase'] = df['verified_purchase'].fillna(False).astype(bool)
        
    df['text'] = df['text'].astype(str).str.strip()
    df = df[df['text'].str.len() >= 10]
    df = df.drop_duplicates(subset=['user_id', 'parent_asin'], keep='first')
    
    return df.reset_index(drop=True)

print(f"Bắt đầu xử lý Review. Chunk size: {CHUNK_SIZE:,}")
chunk_id = 0
total_clean_reviews = 0
current_chunk = []
for f in TEMP_DIR.glob("rev_chunk_*.parquet"): f.unlink()

with open(REVIEW_PATH, 'rt', encoding='utf-8') as f:
    for line in tqdm(f, total=n_reviews, desc="Processing Reviews"):
        try:
            current_chunk.append(json.loads(line.strip()))
        except:
            continue
            
        if len(current_chunk) >= CHUNK_SIZE:
            df_clean = clean_review_chunk(pd.DataFrame(current_chunk))
            if df_clean is not None:
                df_clean.to_parquet(TEMP_DIR / f"rev_chunk_{chunk_id:04d}.parquet", index=False)
                total_clean_reviews += len(df_clean)
            
            chunk_id += 1
            current_chunk = []
            gc.collect()

if current_chunk:
    df_clean = clean_review_chunk(pd.DataFrame(current_chunk))
    if df_clean is not None:
        df_clean.to_parquet(TEMP_DIR / f"rev_chunk_{chunk_id:04d}.parquet", index=False)
        total_clean_reviews += len(df_clean)
    gc.collect()
print("\nĐang gộp các chunks Review...")
review_output = PROCESSED_DIR / "review_clean.parquet"
writer = None
for f in tqdm(sorted(TEMP_DIR.glob("rev_chunk_*.parquet")), desc="Merging Reviews"):
    table = pq.read_table(f)
    if writer is None:
        writer = pq.ParquetWriter(review_output, table.schema, compression='snappy')
    writer.write_table(table)
    f.unlink() 
    
if writer: writer.close()
print(f"Đã lưu: {review_output.name} ({total_clean_reviews:,} dòng)")

Bắt đầu xử lý Review. Chunk size: 200,000


Processing Reviews:   0%|          | 199999/67409944 [00:05<31:14, 35846.56it/s]


KeyboardInterrupt: 

In [13]:

def clean_meta_chunk(df):
    if df.empty: return None

    needed = ['parent_asin', 'title', 'price', 'description', 'categories', 'average_rating', 'main_category', 'images']
    df = df[[c for c in needed if c in df.columns]].copy()

    df = df.dropna(subset=['parent_asin', 'title'])
    df = df.drop_duplicates(subset=['parent_asin'], keep='first')
    def parse_price(p_str):
        p_str = str(p_str).strip()
        if not p_str or p_str == 'nan': return 0.0
        if '-' in p_str:
            try:
                parts = p_str.split('-')
                p1 = float(''.join(c for c in parts[0] if c.isdigit() or c == '.'))
                p2 = float(''.join(c for c in parts[1] if c.isdigit() or c == '.'))
                return (p1 + p2) / 2.0
            except: return 0.0
        try: return float(''.join(c for c in p_str if c.isdigit() or c == '.'))
        except: return 0.0

    if 'price' in df.columns:
        df['price'] = df['price'].apply(parse_price).astype('float32')
    if 'categories' in df.columns:
        def extract_hierarchy(cats):
            if isinstance(cats, list) and len(cats) > 0:
                inner = cats[0] if isinstance(cats[0], list) else cats
                c1 = inner[0] if len(inner) > 0 else 'Unknown'
                c2 = inner[1] if len(inner) > 1 else 'Unknown'
                c3 = inner[2] if len(inner) > 2 else 'Unknown'
                return pd.Series([c1, c2, c3])
            return pd.Series(['Unknown', 'Unknown', 'Unknown'])

        df[['cat_level_1', 'cat_level_2', 'cat_level_3']] = df['categories'].apply(extract_hierarchy)
        df['main_category'] = df['cat_level_1']
        df = df.drop(columns=['categories'])
    for col in ['parent_asin', 'title', 'main_category', 'cat_level_1', 'cat_level_2', 'cat_level_3']:
        if col in df.columns: df[col] = df[col].astype(str)

    return df.reset_index(drop=True)

print(f"\nBắt đầu xử lý Meta. Chunk size: {CHUNK_SIZE:,}")
chunk_id = 0
total_clean_meta = 0
current_chunk = []

for f in TEMP_DIR.glob("meta_chunk_*.parquet"): f.unlink()

with open(META_PATH, 'rt', encoding='utf-8') as f:
    for line in tqdm(f, total=n_meta, desc="Processing Meta"):
        try: current_chunk.append(json.loads(line.strip()))
        except: continue
            
        if len(current_chunk) >= CHUNK_SIZE:
            df_clean = clean_meta_chunk(pd.DataFrame(current_chunk))
            if df_clean is not None:
                df_clean.to_parquet(TEMP_DIR / f"meta_chunk_{chunk_id:04d}.parquet", index=False)
                total_clean_meta += len(df_clean)
            chunk_id += 1
            current_chunk = []
            gc.collect()

if current_chunk:
    df_clean = clean_meta_chunk(pd.DataFrame(current_chunk))
    if df_clean is not None:
        df_clean.to_parquet(TEMP_DIR / f"meta_chunk_{chunk_id:04d}.parquet", index=False)
        total_clean_meta += len(df_clean)

print("\nĐang gộp các chunks Meta...")
meta_output = PROCESSED_DIR / "meta_clean.parquet"
writer = None
for f in tqdm(sorted(TEMP_DIR.glob("meta_chunk_*.parquet")), desc="Merging Meta"):
    table = pq.read_table(f)
    if writer is None: writer = pq.ParquetWriter(meta_output, table.schema, compression='snappy')
    writer.write_table(table)
    f.unlink()
    
if writer: writer.close()
print(f"Đã lưu: {meta_output.name} ({total_clean_meta:,} dòng)")


Bắt đầu xử lý Meta. Chunk size: 200,000


Processing Meta: 100%|██████████| 3735584/3735584 [11:21<00:00, 5485.05it/s] 



Đang gộp các chunks Meta...


Merging Meta: 100%|██████████| 19/19 [00:29<00:00,  1.56s/it]

Đã lưu: meta_clean.parquet (3,735,584 dòng)
